# Big Data Platforms — Lecture 10 Notes

Course: DATA140031 (MOOC, 3 ECTS) + DATA140032 (MOOC EXAM, 2 ECTS)
Lecturer: Keijo Heljanko, Department of Computer Science, University of Helsinki
Date: 8.10.2026

Lecture 10 is the "putting it all together" lecture. Lectures 3–9 took the
cloud apart piece by piece: Spark and MapReduce, HDFS, RAID and erasure codes,
the storage hierarchy, CAP and consensus, BigTable, Dynamo. This one steps back
and asks where all those pieces sit in a real cloud stack, and which open-source
projects fill each slot. It then adds three things the earlier lectures only
hinted at: object storage (S3) and why it forces a particular design on
anything built on top of it, the Delta Lake and Lambda architecture patterns for
getting reliable results out of unreliable parts, and probabilistic data
structures as a way to answer big-data questions approximately but cheaply.

One housekeeping note from the first content slide: **lecture 11, the last one,
will cover the department's own research on Big Data processing.**


## 1. Cloud and open source — the cloning pattern

The lecture opens with an observation about how open-source infrastructure tends
to come about: **successful open-source projects often start as clones of
existing proprietary systems.** HDFS cloned GFS, Hadoop MapReduce cloned Google
MapReduce, HBase cloned BigTable, ZooKeeper cloned Chubby — every one of those
has appeared in the previous lectures. So a practical way to map the open-source
cloud landscape is to look at what the proprietary industry leaders actually
run, then find the open-source counterpart for each component.

Two leaders get that treatment: Amazon (the leading commercial IaaS provider)
and Google (the leading SaaS provider).


## 2. What's inside the Amazon Web Services stack

AWS is organized into a long list of top-level categories:

- Compute & Containers
- Databases
- Application Integration (messaging)
- Security, Identity & Compliance, Crypto & PKI
- Machine Learning & Analytics
- Management & Governance
- Networking & Content Delivery, Media Services
- IoT, Web & Mobile
- Development Tools, Applications
- ...and many more

Many of these categories aren't software at all but other services and
hardware, so they have no open-source replacement to speak of. Others do, and
several of those replacements are under heavy development.

### Open-source counterparts the lecture names

| Amazon service | What it does | Open-source counterpart |
|---|---|---|
| EC2 | Virtual machines | **OpenStack Compute** |
| (scalable batch processing) | Large-scale batch jobs | **Apache Hadoop MapReduce** (a Google MapReduce clone) and **Apache Spark** |
| EBS | Local-network block storage | **Ceph Block Device** |
| S3 | Global object storage | **Ceph Object Gateway**, **OpenStack Object Storage (Swift)** |
| Kinesis | Event streaming | **Apache Kafka** |

### Amazon also uses open source directly

Amazon doesn't only clone things; it also runs open-source systems itself, and
sometimes **forks** them to get around licence restrictions. The lecture's
examples:

- **Elasticsearch and Kibana** — Amazon's fork is called **OpenSearch**.
- **MongoDB**
- **Amazon Elastic MapReduce (EMR)** — built on the Hadoop/Spark ecosystem.


## 3. Open-sourcing the datacenter itself

The idea goes below software too. **Open Compute** (<http://opencompute.org/>)
started at Facebook and is now a very large consortium. It publishes fully
open-source designs for the physical datacenter:

- A datacenter design that leans heavily on **outside-air cooling**.
- **Custom server designs** for Intel and AMD processors.
- Designs for **electrical systems and distributed UPSs** (the same
  uninterruptible power supplies lecture 4 kept recommending for RAID write-hole
  protection, here designed in at the datacenter level).
- All of it fully open sourced.


## 4. Google's infrastructure and its open-source replacements

Google is the leading SaaS cloud provider, and nearly every piece of its
infrastructure has been described in a paper that someone then cloned:

| Google system | Role | Open-source replacement(s) |
|---|---|---|
| **GFS** (Google File System) | Distributed filesystem | **HDFS** |
| **MapReduce** | Distributed batch processing | **Hadoop MapReduce**, then **Apache Spark** |
| **Bigtable / Spanner / Dremel** | Distributed databases | **HBase**, **Apache Cassandra**, **Apache Hive**, **Cloudera Impala**, **Spark SQL**, **Facebook Presto** |
| **Chubby** | Distributed coordination service | **Apache ZooKeeper**, **etcd** (e.g. in Kubernetes) |

Many of the replacements were developed by *other* web companies — Facebook,
LinkedIn, Twitter and so on — not by Google.

### How Google's stack is built

- It's a **massively distributed, fault-tolerant PaaS** underneath Google's SaaS
  products.
- It runs on **commodity hardware**, the same philosophy as HDFS in lecture 3.
- The design focus is **automated fault tolerance and multi-tenancy**, to keep
  the number of computers each administrator has to look after as high as
  possible.
- Google has the **"luxury of embarrassing parallelism"**: running *one* word
  processor is inherently sequential, but running *a million* word processors
  at once is trivially parallel — every user's session is independent of every
  other's. That's a big part of why this style of architecture works so well
  for SaaS.


## 5. Cloud storage — a taxonomy

Clouds use several quite different kinds of storage, and picking the right one
mostly comes down to what you need from it. The lecture goes through five.

### Node-local hard disks
- Typically used for **temporary data and binaries**.
- High performance, but storage tends to be **over-provisioned**: spare space on
  one node can't easily be lent to another.
- Even with RAID inside, the **server itself is a single point of failure**.

### Network block devices
- Examples: **Amazon EBS**, **Ceph block device**.
- Virtual block devices reached over the network, often with **RAID 1-like
  durability** (every block mirrored).
- Many systems only allow each block device to be **mounted by one client at a
  time**. That keeps things simple and lets the system scale by having
  different storage servers serve different clients.
- Space is used **much more efficiently** than with node-local disks, at the
  price of needing a **fault-tolerant management layer** that maps physical
  storage onto the virtual devices clients see.
- More traditional, less automated alternatives: **Linux DRBD** (Distributed
  Replicated Block Device) and **SAN storage over iSCSI**.

### POSIX filesystems
- Examples: **NFS**, **Lustre** (almost POSIX).
- Shared filesystems used by several clients at once.
- **Limited scalability**, because the metadata of a shared namespace has to be
  coordinated between clients and metadata updates go through a single server.
- Because of that, they're **rarely used in cloud applications**.

### Distributed WORM filesystems
- Examples: **GFS**, **HDFS**.
- Built for **big files and write-once-read-many workloads**, and very scalable
  in exactly that scenario.
- Both were originally limited by a **single metadata server** (the NameNode in
  HDFS). The usual workaround is **sharding the namespace across several
  metadata servers**, which is the HDFS Federation idea from lecture 3.
- **HopsFS** goes further and replaces the NameNode with a **distributed
  database** to scale metadata handling.
- **Windows Azure Blob Storage** is a proprietary system with an HDFS-compatible
  API.

### Object storage
- Examples: **Amazon S3**, **OpenStack Swift** (developed by Rackspace),
  **Ceph Object Storage**.
- **No nested directory hierarchy.** Objects live in **buckets**, and each item
  is reached through a **unique identifier**.
- **Extremely scalable**, because each object can be served by a different
  server, chosen for example by hashing its identifier (the same family of idea as
  lecture 9's consistent hashing).
- Usually accessed over an **HTTP REST API**, with the **Amazon S3 API** being
  the most common choice (so even the open-source systems tend to speak it).


## 6. Amazon S3 in detail

- S3 is **geographically replicated**: each data item is stored in **at least
  two geographically remote datacenters** by default.
- It's designed for **99.999999999% durability** and **99.99% availability** of
  objects over a given year.
- Since **December 2020** it is **"strongly consistent"** for reads after
  writes (<https://aws.amazon.com/blogs/aws/amazon-s3-update-strong-read-after-write-consistency/>).

**But** — and this is the key design point of the whole section — two
**concurrent writes** to the same object still follow **last-write-wins
semantics** based on timestamps, and S3 offers **no locking and no multi-key
transactions.** In lecture 6/9's terms that makes S3 essentially an **AP**
design.

The consequence the lecture draws: S3 doesn't have the CP features you'd need
to build a CP database *directly* on top of it, so **the way to use it is as an
AP, write-once-read-many store.** That's the same WORM discipline HDFS imposes,
and it's what the next section builds on.


In [1]:
# What do "eleven nines" of durability and "four nines" of availability
# actually mean in practice? The slide quotes the percentages; here's the
# arithmetic behind them.

durability = 0.99999999999          # 99.999999999% per object per year
availability = 0.9999               # 99.99% over a year

annual_loss_probability = 1 - durability
print(f"Per-object annual loss probability: {annual_loss_probability:.0e}")

for n_objects in [1_000_000, 10_000_000, 1_000_000_000]:
    expected_losses_per_year = n_objects * annual_loss_probability
    years_per_loss = 1 / expected_losses_per_year
    print(f"  {n_objects:>13,} objects -> expected losses per year: "
          f"{expected_losses_per_year:.1e}  (about one every {years_per_loss:,.0f} years)")

minutes_per_year = 365.25 * 24 * 60
downtime_minutes = (1 - availability) * minutes_per_year
print(f"\n99.99% availability allows about {downtime_minutes:.1f} minutes "
      f"(~{downtime_minutes/60:.2f} hours) of unavailability per year")


Per-object annual loss probability: 1e-11
      1,000,000 objects -> expected losses per year: 1.0e-05  (about one every 100,000 years)
     10,000,000 objects -> expected losses per year: 1.0e-04  (about one every 10,000 years)
  1,000,000,000 objects -> expected losses per year: 1.0e-02  (about one every 100 years)

99.99% availability allows about 52.6 minutes (~0.88 hours) of unavailability per year


Durability and availability are different promises. Eleven nines of durability
says the object will almost certainly still *exist* a year from now; four nines of
availability says you can *reach* it for all but roughly an hour of that year.
An object can be temporarily unreachable without being lost, and the slide's
numbers reflect that: the durability figure is seven orders of magnitude stricter
than the availability figure.


## 7. Data warehouse curation — bronze, silver, gold

Data warehouses commonly organize data into **three tiers**, a pattern sometimes
called the *medallion* architecture and used by Delta Lake:

- **Bronze** — **raw ingested data**, including its history.
- **Silver** — **filtered, cleaned, and augmented** data.
- **Gold** — **business-level aggregates and machine-learning features.**

The slide gives the reason for keeping the bronze layer around: **keeping
historical data matters because you will eventually find bugs in your cleaning
or aggregation logic.** If only the cleaned data were kept, a cleaning bug would
be permanent; with the raw data retained, you fix the bug and recompute silver
and gold from bronze.


### Delta Lake medallion architecture (bronze / silver / gold)

![Delta Lake medallion architecture (bronze / silver / gold)](images/delta_lake.png)


In [2]:
# A tiny bronze -> silver -> gold pipeline, to make the three tiers concrete.
# Toy data: raw page-view events, some of them malformed or duplicated.

bronze = [   # raw, exactly as ingested -- never modified, history kept
    {"event_id": 1, "user": "alice", "country": "FI", "ms": "120"},
    {"event_id": 2, "user": "bob",   "country": "fi", "ms": "250"},
    {"event_id": 2, "user": "bob",   "country": "fi", "ms": "250"},   # duplicate delivery
    {"event_id": 3, "user": "carol", "country": "SE", "ms": "oops"},  # corrupt value
    {"event_id": 4, "user": "dave",  "country": "SE", "ms": "80"},
]

def to_silver(bronze_rows):
    seen, silver = set(), []
    for row in bronze_rows:
        if row["event_id"] in seen:
            continue                                # drop duplicates
        try:
            ms = int(row["ms"])                     # drop rows we can't parse
        except ValueError:
            continue
        seen.add(row["event_id"])
        silver.append({**row, "country": row["country"].upper(), "ms": ms})
    return silver

def to_gold(silver_rows):
    gold = {}
    for row in silver_rows:
        agg = gold.setdefault(row["country"], {"views": 0, "total_ms": 0})
        agg["views"] += 1
        agg["total_ms"] += row["ms"]
    return {c: {**a, "avg_ms": a["total_ms"] / a["views"]} for c, a in gold.items()}

silver = to_silver(bronze)
gold = to_gold(silver)
print("silver rows:", len(silver), "(from", len(bronze), "bronze rows)")
print("gold:", gold)

# Suppose we later discover a cleaning bug: 'oops' rows should have been
# repaired to a default of 100 ms, not dropped. Because bronze was kept, we
# just fix the rule and recompute -- nothing is lost.
def to_silver_fixed(bronze_rows):
    seen, silver = set(), []
    for row in bronze_rows:
        if row["event_id"] in seen:
            continue
        try:
            ms = int(row["ms"])
        except ValueError:
            ms = 100                                # the corrected rule
        seen.add(row["event_id"])
        silver.append({**row, "country": row["country"].upper(), "ms": ms})
    return silver

print("\nAfter fixing the cleaning rule and recomputing from bronze:")
print("gold:", to_gold(to_silver_fixed(bronze)))


silver rows: 3 (from 5 bronze rows)
gold: {'FI': {'views': 2, 'total_ms': 370, 'avg_ms': 185.0}, 'SE': {'views': 1, 'total_ms': 80, 'avg_ms': 80.0}}

After fixing the cleaning rule and recomputing from bronze:
gold: {'FI': {'views': 2, 'total_ms': 370, 'avg_ms': 185.0}, 'SE': {'views': 2, 'total_ms': 180, 'avg_ms': 90.0}}


## 8. Delta Lake — ACID tables on top of object storage

Delta Lake is a data-warehousing framework that implements **ACID transactions**
(lecture 6's A-C-I-D) for tables stored on systems that don't natively provide
them.

- It lets you build **database tables with transactions on top of an object
  store**: **S3, Azure Blob Storage, Azure Data Lake Storage Gen2**, and also
  **HDFS**.
- It can **integrate with streaming systems**. The minimum latency depends on
  the underlying object store but is **on the order of several seconds** — so
  this is near-real-time, not millisecond-real-time.
- It can **manage and look up table partitions** stored as S3 objects or HDFS
  files, which **reduces the metadata load** on those services.
- It can provide **GDPR data-removal functionality and audit logs.**

### The S3 catch

Remember from section 6: S3 gives last-write-wins on concurrent writes and no
locking. Delta Lake's transactions depend on being able to commit atomically,
so with S3 there's a real problem:

- Delta Lake **needs an external locking service** (essentially a database) to
  manage concurrent writes to S3 objects. The lecture notes this is available
  from the **Databricks commercial version**.
- The documentation's own warning, quoted on the slide: **concurrent writes to
  the same Delta table from multiple Spark drivers can lead to data loss.**
  (<https://docs.delta.io/latest/delta-storage.html#amazon-s3>)
- **Many other supported storage systems don't have this problem**, because
  they are typically **CP systems** and can provide the needed atomicity
  themselves.

The next cell is a deliberately simplified model of why this happens. It
treats a table as a numbered log of commits, with each writer committing by
writing "version N+1". It is **not** Delta Lake's actual code; it only shows
the failure mode the slide warns about.


In [3]:
# Simplified model: a table is a log of commits stored as numbered "files"
# in an object store. A writer commits by writing the NEXT version number.
#
# Store A behaves like plain S3 for concurrent writes: last write wins, no
# locking, so two writers who both decide to write version 2 silently clobber
# each other. Store B adds an atomic "put if absent" (what an external locking
# service provides), so the second writer is told it lost the race.

class LastWriteWinsStore:
    def __init__(self):
        self.files = {}
    def put(self, name, content):
        self.files[name] = content           # silently overwrites anything there
        return True

class AtomicStore(LastWriteWinsStore):
    def put(self, name, content):
        if name in self.files:               # atomic check-and-set
            return False                     # lost the race: caller must retry
        self.files[name] = content
        return True

def commit(store, writer, change, read_version):
    """Optimistic commit: write version read_version+1; retry on conflict."""
    attempt_version = read_version + 1
    while True:
        ok = store.put(f"v{attempt_version:03d}", f"{writer}: {change}")
        if ok:
            return attempt_version
        attempt_version += 1                 # someone else won; try the next slot

# Both writers read the table when it was at version 1, then each commit once.
for label, store_cls in [("Plain S3-like store (last write wins)", LastWriteWinsStore),
                         ("Store with atomic put-if-absent",       AtomicStore)]:
    store = store_cls()
    store.put("v001", "initial load")
    v_a = commit(store, "writer-A", "add rows 1-100", read_version=1)
    v_b = commit(store, "writer-B", "add rows 101-200", read_version=1)
    print(label)
    for name in sorted(store.files):
        print(f"   {name}: {store.files[name]}")
    committed = {v_a, v_b}
    print(f"   writer-A committed as v{v_a:03d}, writer-B as v{v_b:03d}")
    lost = [w for w in ("writer-A", "writer-B")
            if not any(w in c for c in store.files.values())]
    print(f"   commits lost: {lost if lost else 'none'}\n")


Plain S3-like store (last write wins)
   v001: initial load
   v002: writer-B: add rows 101-200
   writer-A committed as v002, writer-B as v002
   commits lost: ['writer-A']

Store with atomic put-if-absent
   v001: initial load
   v002: writer-A: add rows 1-100
   v003: writer-B: add rows 101-200
   writer-A committed as v002, writer-B as v003
   commits lost: none



In the last-write-wins store, writer-B's commit lands on the same slot as
writer-A's and replaces it. Writer-A was told its commit succeeded, but its
change is gone from the log: that's the "concurrent writes can lead to data loss"
warning, reproduced. With an atomic check-and-set the second writer finds the
slot taken, retries at the next version, and both commits survive. This is also
why lecture 6's coordination services (ZooKeeper, etcd) keep reappearing: when
the storage layer can't give you an atomic primitive, you borrow one from a CP
system.


## 9. Stream processing frameworks

**Why they exist.** One motivation for the Lambda architecture (section 11) was
that early **stream layers lacked fault tolerance**. Newer streaming frameworks
have fault tolerance built in:

- **Spark Structured Streaming** — Spark extended to stream processing.
- **Apache Flink** — an open-source, **low-latency** streaming framework.
- **Google Cloud Dataflow**
- **Azure Stream Analytics**
- **Apache Beam** — an open-source programming framework for **both batch and
  streaming**, originally created by Google. It has **runners** for Google Cloud
  Dataflow, Spark, Flink and others, so the same pipeline definition can run on
  different engines.

Stream processors also need somewhere reliable to read events from and write
them to: a **fault-tolerant publish/subscribe messaging system**. The lecture
lists **Apache Kafka** (open source), **Amazon Kinesis**, and **Azure Event
Hub**.


## 10. Apache Kafka

Kafka is an open-source, fault-tolerant **publish/subscribe messaging
platform**. The design points the lecture highlights:

- **Coordination data lives elsewhere.** Slowly changing management and
  configuration data is stored in **ZooKeeper** or, since **Kafka 3.5.0**, in
  **KRaft** (Kafka's own Raft-based replacement for ZooKeeper). Kubernetes' use
  of **etcd** plays the same role in its own world.
- **A custom replication algorithm**, "similar in spirit to Paxos, Zab, and Raft
  but differing in technical details" — lecture 6's family of protocols, with
  Kafka's own variations.
- **Many small queues instead of one big one.** A topic is split into a very
  large number of pub/sub queues called **partitions**, and **each partition has
  its own leader.** Lecture 6 pointed out that routing all writes through a
  single leader makes that leader's throughput the system's ceiling. Kafka
  avoids the ceiling by having *lots* of leaders, each responsible for a slice
  of the data.
- The same trick is used by many **sharded distributed databases** — the lecture
  names **Apache Kudu, CockroachDB, and Google Spanner**.
- The lecture is careful to add that this **does not solve the global
  transactions problem**: sharding gives you scalable *per-shard* ordering, not
  a consistent order across shards.


In [4]:
# Many partitions, each with its own leader, spread over a few brokers:
# the leader bottleneck from lecture 6 goes away because no single node
# handles all the writes.

import hashlib
from collections import Counter

brokers = ["broker-1", "broker-2", "broker-3", "broker-4"]
num_partitions = 12

# Partition leaders are spread round-robin over the brokers
leader_of = {p: brokers[p % len(brokers)] for p in range(num_partitions)}

def partition_for(key):
    return int(hashlib.md5(key.encode()).hexdigest(), 16) % num_partitions

keys = [f"user-{i}" for i in range(10_000)]
writes_per_broker = Counter(leader_of[partition_for(k)] for k in keys)

print(f"{len(keys):,} writes spread over {num_partitions} partitions "
      f"and {len(brokers)} brokers:")
for b in brokers:
    share = writes_per_broker[b] / len(keys)
    print(f"  {b}: {writes_per_broker[b]:>5} writes  ({share:.1%})")

print(f"\nWith a SINGLE leader, one node would take 100% of the writes.")
print("Note: records with the same key always land in the same partition, so")
print("ordering is preserved per key -- but there is no global order across partitions.")


10,000 writes spread over 12 partitions and 4 brokers:
  broker-1:  2378 writes  (23.8%)
  broker-2:  2539 writes  (25.4%)
  broker-3:  2553 writes  (25.5%)
  broker-4:  2530 writes  (25.3%)

With a SINGLE leader, one node would take 100% of the writes.
Note: records with the same key always land in the same partition, so
ordering is preserved per key -- but there is no global order across partitions.


## 11. The Lambda architecture

**The tension it resolves:**

- **Batch systems** (MapReduce, Spark in batch mode) have **high latency —
  minutes to hours — but excellent fault tolerance**: each data item is
  processed **exactly once**. That's the **CP** side.
- Sometimes results are needed **in real time**. **Stream processors** such as
  **Apache Storm** can process data at scale with **latencies of tens of
  milliseconds**, but with **minimal fault tolerance**: each item is processed
  **at least once**, so duplicates are possible. That's the **AP** side.

The **Lambda architecture** combines the two: **consistent, exact, high-latency
batch processing** alongside **available, approximate, low-latency stream
processing**. Details: <https://en.wikipedia.org/wiki/Lambda_architecture>

### The three layers

- **Batch layer** — **exact computation, high latency**. Maintains the **master
  dataset**. A **CP** system.
- **Speed layer** — **approximate computation, low latency, minimal fault
  tolerance**. Only maintains approximations over **recent data the batch layer
  hasn't processed yet.** An **AP** system.
- **Serving layer** — serves the output of both layers to queries.

The payoff is what the slide calls **"eventual accuracy"**: the speed layer's
approximations are **eventually overwritten by the exact results** from the
batch layer. Users get a fast, roughly right answer immediately and an exactly
right one later.


### Lambda architecture (batch, speed, and serving layers)

![Lambda architecture (batch, speed, and serving layers) — placeholder](images/lambda_architecture.png)



In [5]:
# A toy Lambda architecture: counting events per page.
#   batch layer  : exact, but only covers data up to the last batch run
#   speed layer  : fast, covers only events since the last batch run, and is
#                  AT-LEAST-ONCE, so it can over-count duplicates
#   serving layer: batch view + speed view

import random
random.seed(3)

master_dataset = []     # the batch layer's immutable master dataset
batch_view = {}         # exact counts as of the last batch run
speed_view = {}         # approximate counts for events since the last batch run

def ingest(event):
    master_dataset.append(event)                       # kept forever, exactly once
    speed_view[event] = speed_view.get(event, 0) + 1   # fast path
    if random.random() < 0.15:                         # at-least-once: sometimes redelivered
        speed_view[event] += 1

def run_batch():
    global batch_view, speed_view
    batch_view = {}
    for event in master_dataset:                       # recompute exactly from scratch
        batch_view[event] = batch_view.get(event, 0) + 1
    speed_view = {}                                    # approximations are overwritten

def query(page):
    return batch_view.get(page, 0) + speed_view.get(page, 0)

pages = ["/home", "/news", "/sports"]
for _ in range(200):
    ingest(random.choice(pages))

exact = {p: master_dataset.count(p) for p in pages}
print("Before the batch layer catches up (serving = batch + speed):")
for p in pages:
    print(f"  {p:<8} served = {query(p):>3}   exact = {exact[p]:>3}   "
          f"error = {query(p) - exact[p]:+d}")

run_batch()
print("\nAfter the batch run overwrites the approximations:")
for p in pages:
    print(f"  {p:<8} served = {query(p):>3}   exact = {exact[p]:>3}   "
          f"error = {query(p) - exact[p]:+d}")


Before the batch layer catches up (serving = batch + speed):
  /home    served =  85   exact =  72   error = +13
  /news    served =  62   exact =  54   error = +8
  /sports  served =  86   exact =  74   error = +12

After the batch run overwrites the approximations:
  /home    served =  72   exact =  72   error = +0
  /news    served =  54   exact =  54   error = +0
  /sports  served =  74   exact =  74   error = +0


## 12. Approximation with probabilistic data structures

The final theme ties back to lecture 7's Bloom filters. Some questions are
expensive to answer exactly. The lecture's example: **the number of distinct
users who visited a website.** Answering that exactly means looking at *all* the
data, and with new data arriving constantly, keeping the number current means
**recomputing it** again and again.

**Probabilistic data structures** answer such queries **approximately** instead,
and as a bonus they can make the computation **parallel and fully incremental**.
That's the same memory-for-accuracy trade as Bloom filters, applied to counting
instead of set membership.

### In Spark

- The exact `distinct().count()` DataFrame operation can be replaced by
  **`approx_count_distinct()`**.
- Internally it uses the **HyperLogLog++** probabilistic data structure, which
  **minimizes the data shuffled over the network** (shuffle cost is the thing
  lecture 3 said to minimize), and therefore speeds queries up.
- Spark supports other approximate computations built on probabilistic
  structures: **approximate quantiles, frequent items, frequency estimation
  with a count-min sketch, Bloom filters, and stratified sampling.**
- Documentation: <https://spark.apache.org/docs/latest/api/java/org/apache/spark/sql/DataFrameStatFunctions.html>


In [6]:
# Using it in PySpark. Guarded, because this needs pyspark and a working Java
# runtime; if either is missing the cell just says so instead of failing.

try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F

    spark = SparkSession.builder.master("local[2]").appName("approx-demo").getOrCreate()
    df = spark.range(0, 200_000).withColumn("user", (F.col("id") % 50_000))

    exact = df.select("user").distinct().count()
    approx = df.select(F.approx_count_distinct("user")).first()[0]
    print(f"exact distinct users:  {exact:,}")
    print(f"approx_count_distinct: {approx:,}  "
          f"(error {abs(approx - exact) / exact:.2%})")
    spark.stop()
except Exception as e:
    print("Skipping the Spark demo (pyspark/Java not available here):", type(e).__name__)


C:\Users\biswa\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


exact distinct users:  50,000
approx_count_distinct: 50,113  (error 0.23%)


### What HyperLogLog actually does

HyperLogLog (HLL) rests on a neat observation. Hash every item to a uniformly
random bit string. If you've seen *n* distinct items, then the **longest run of
leading zeros** in any of their hashes is about log₂(n), because a run of *k*
leading zeros shows up with probability 2⁻ᵏ. A single such maximum is a noisy
estimate, so HLL splits items into *m* buckets (by the first few hash bits),
keeps the maximum run length for each bucket in a tiny register, and combines
the registers with a harmonic mean. With *m* registers the typical relative
error is about **1.04/√m**, and memory is only *m* small numbers, no matter how
many distinct items go through.

The version below is a plain HLL for illustration; Spark's HyperLogLog++ adds
bias corrections and a sparse representation for small counts.


In [7]:
import hashlib
import math

class HyperLogLog:
    def __init__(self, p=10):
        self.p = p
        self.m = 1 << p                       # number of registers (buckets)
        self.registers = [0] * self.m
        # standard bias-correction constant for m >= 128
        self.alpha = 0.7213 / (1 + 1.079 / self.m)

    def _hash64(self, item):
        return int.from_bytes(hashlib.sha1(str(item).encode()).digest()[:8], "big")

    def add(self, item):
        x = self._hash64(item)
        bucket = x >> (64 - self.p)                         # first p bits pick the register
        rest = x & ((1 << (64 - self.p)) - 1)               # remaining bits
        # position of the first 1-bit in the remaining (64-p) bits
        rank = (64 - self.p) - rest.bit_length() + 1
        self.registers[bucket] = max(self.registers[bucket], rank)

    def estimate(self):
        z = 1.0 / sum(2.0 ** -r for r in self.registers)
        e = self.alpha * self.m * self.m * z
        zeros = self.registers.count(0)
        if e <= 2.5 * self.m and zeros:                      # small-range correction
            e = self.m * math.log(self.m / zeros)
        return e

    def merge(self, other):
        merged = HyperLogLog(self.p)
        merged.registers = [max(a, b) for a, b in zip(self.registers, other.registers)]
        return merged


hll = HyperLogLog(p=10)                       # 1024 registers
true_distinct = 100_000
for i in range(true_distinct):
    hll.add(f"user-{i}")
    hll.add(f"user-{i}")                      # duplicates don't change the answer

est = hll.estimate()
print(f"true distinct: {true_distinct:,}")
print(f"HLL estimate:  {est:,.0f}   (error {abs(est - true_distinct) / true_distinct:.2%})")
print(f"expected relative error ~ 1.04/sqrt({hll.m}) = {1.04 / math.sqrt(hll.m):.2%}")
print(f"memory: {hll.m} small registers, versus {true_distinct:,} ids for an exact set")


true distinct: 100,000
HLL estimate:  103,109   (error 3.11%)
expected relative error ~ 1.04/sqrt(1024) = 3.25%
memory: 1024 small registers, versus 100,000 ids for an exact set


## 13. Incremental computation — sketches, monoids, and Algebird

The last slide pulls the pieces together into a general pattern, supported by
**Algebird**, a **Scala** library for incremental computation that can be
combined with Spark.

The idea:

1. **Partition the data into batches.**
2. **Summarize each batch** (sequentially or in parallel) into a small
   probabilistic structure called a **sketch**.
3. **Combine all the sketches into one**, which can answer the query.
4. The combined sketch can also be **updated incrementally** as new batches
   arrive, without reprocessing the old data.

For this to work, **the operation that combines sketches has to be
associative**: you must be able to combine them in any grouping and get the same
answer, because a parallel system will group them however is convenient.
Algebird calls data representations with an associative combine operation (and
an identity element) **Monoids**. Resources:
<https://twitter.github.io/algebird/resources_for_learners.html>

This connects back to lecture 9: a CRDT's merge is the same sort of thing (and
is additionally commutative and idempotent). It also connects to lecture 3's
combiners, which need an associative, commutative reduce function. Here is the
property checked on HLL sketches.


In [8]:
import random

# Build one sketch per "batch" of data, as a parallel system might.
def sketch_of(items):
    h = HyperLogLog(p=10)
    for it in items:
        h.add(it)
    return h

random.seed(11)
all_users = [f"user-{i}" for i in range(60_000)]
batch1, batch2, batch3 = all_users[:20_000], all_users[20_000:40_000], all_users[40_000:]
s1, s2, s3 = sketch_of(batch1), sketch_of(batch2), sketch_of(batch3)

# Associativity: grouping doesn't matter
left  = s1.merge(s2).merge(s3)
right = s1.merge(s2.merge(s3))
print("(s1+s2)+s3 == s1+(s2+s3):", left.registers == right.registers)

# Commutativity: order doesn't matter
print("s1+s2 == s2+s1:          ", s1.merge(s2).registers == s2.merge(s1).registers)

# Identity ("zero") element: an empty sketch
zero = HyperLogLog(p=10)
print("s1+zero == s1:           ", s1.merge(zero).registers == s1.registers)

# Merging batch sketches equals sketching all the data in one go
whole = sketch_of(all_users)
print("merged sketches == one sketch over everything:", left.registers == whole.registers)
print(f"\nestimate from merged sketches: {left.estimate():,.0f}   (true: {len(all_users):,})")

# Incremental update: a new batch arrives later -- no need to touch old data
new_batch = [f"user-{i}" for i in range(60_000, 70_000)]
updated = left.merge(sketch_of(new_batch))
print(f"after a new batch of 10,000 users: {updated.estimate():,.0f}   (true: 70,000)")


(s1+s2)+s3 == s1+(s2+s3): True
s1+s2 == s2+s1:           True
s1+zero == s1:            True
merged sketches == one sketch over everything: True

estimate from merged sketches: 60,983   (true: 60,000)
after a new batch of 10,000 users: 69,866   (true: 70,000)


That's the whole trick. Because merging is associative (with an identity
element), the batch layer of a Lambda architecture can summarize each batch in
parallel on different machines, the speed layer can keep a sketch of just the
recent data, and the serving layer can combine the two sketches at query time.
A "count distinct users" query then costs a few kilobytes of sketch instead of a
scan of the raw data.


## Summary

Lecture 10 is a map of the cloud, and a few warnings about reading it.

- The **open-source cloud largely follows the proprietary one**: for each AWS or
  Google component there's a clone (OpenStack, Ceph, Swift, Kafka, HDFS, Spark,
  HBase, ZooKeeper, etcd), and even the datacenter hardware has an open design
  (Open Compute).
- **Storage comes in five flavours** — node-local disks, network block devices,
  POSIX filesystems, distributed WORM filesystems, and object stores — each with
  a different scalability and consistency profile. Object stores like S3 scale
  furthest, but they are essentially **AP**: last-write-wins, no locking, no
  multi-key transactions. So you should treat them as write-once-read-many.
- Tools like **Delta Lake** layer ACID tables on top of such stores, but only
  work safely if some component can provide an atomic commit, which on S3
  means an external locking service.
- **Kafka** gets its scalability from many partitions with many leaders, and
  the **Lambda architecture** gets low latency *and* exact results by pairing a
  CP batch layer with an AP speed layer, so that approximate answers are
  eventually overwritten by exact ones.
- **Probabilistic data structures** (HyperLogLog++, count-min sketch, Bloom
  filters) supply the approximations cheaply, and because their combine operation
  is associative they parallelize and update incrementally, which is exactly
  what a Lambda-style system needs.

The through-line of the whole course shows up again here: at this scale you
rarely get a guarantee for free. You pick which one to give up (consistency,
latency, exactness, memory), and then build the rest of the system around that
choice.
